# Sequência de Fibonacci

## Recursivo comum

In [1]:
import pandas as pd

class ContadorFib:
    def __init__(self):
        self.chamadas = 0


def fib_recursivo(n, contador=None):
    if contador is None:
        contador = ContadorFib()

    contador.chamadas += 1

    if n <= 1:
        return n, contador

    fib1, contador = fib_recursivo(n - 1, contador)
    fib2, contador = fib_recursivo(n - 2, contador)
    return fib1 + fib2, contador


## Tabela recursivo comum

In [2]:
linhas = []
for n in range(0, 11):
    valor, contador = fib_recursivo(n)
    linhas.append({'N': n, 'Fib(N)': valor, 'Total de chamadas': contador.chamadas})

display(pd.DataFrame(linhas))


,N,Fib(N),Total de chamadas
0,0,0,1
1,1,1,1
2,2,1,3
3,3,2,5
4,4,3,9
5,5,5,15
6,6,8,25
7,7,13,41
8,8,21,67
9,9,34,109


## Top-down com memoização

In [3]:
class ContadorFibMemo:
    def __init__(self):
        self.chamadas = 0
        self.calculos = 0
        self.reusos = 0
        self.estados = []


def fib_top_down(n, memo=None, contador=None):
    if memo is None:
        memo = [None] * (n + 1)
    if contador is None:
        contador = ContadorFibMemo()

    contador.chamadas += 1

    if memo[n] is not None:
        contador.reusos += 1
        contador.estados.append({'n_solicitado': n, 'ação': 'reuso', 'memo': memo.copy()})
        return memo[n], memo, contador

    contador.calculos += 1

    if n <= 1:
        memo[n] = n
    else:
        v1, memo, contador = fib_top_down(n - 1, memo, contador)
        v2, memo, contador = fib_top_down(n - 2, memo, contador)
        memo[n] = v1 + v2

    contador.estados.append({'n_solicitado': n, 'ação': 'cálculo', 'memo': memo.copy()})
    return memo[n], memo, contador


## Tabela top-down

In [4]:
linhas = []
for n in range(0, 11):
    valor, memo, contador = fib_top_down(n)
    linhas.append({
        'N': n,
        'Fib(N)': valor,
        'Total de chamadas': contador.chamadas,
        'Cálculos reais': contador.calculos,
        'Reusos do memo': contador.reusos
    })

display(pd.DataFrame(linhas))


,N,Fib(N),Total de chamadas,Cálculos reais,Reusos do memo
0,0,0,1,1,0
1,1,1,1,1,0
2,2,1,3,3,0
3,3,2,5,4,1
4,4,3,7,5,2
5,5,5,9,6,3
6,6,8,11,7,4
7,7,13,13,8,5
8,8,21,15,9,6
9,9,34,17,10,7


## Tabela de valores salvos no memo

In [5]:
n = 6
valor, memo, contador = fib_top_down(n)

print('Fib(6):', valor)
display(pd.DataFrame(contador.estados))

tabela_memo = pd.DataFrame({'Índice': list(range(len(memo))), 'Valor salvo': memo})
display(tabela_memo)


Fib(6): 8


,n_solicitado,ação,memo
0,1,cálculo,"[None, 1, None, None, None, None, None]"
1,0,cálculo,"[0, 1, None, None, None, None, None]"
2,2,cálculo,"[0, 1, 1, None, None, None, None]"
3,1,reuso,"[0, 1, 1, None, None, None, None]"
4,3,cálculo,"[0, 1, 1, 2, None, None, None]"
5,2,reuso,"[0, 1, 1, 2, None, None, None]"
6,4,cálculo,"[0, 1, 1, 2, 3, None, None]"
7,3,reuso,"[0, 1, 1, 2, 3, None, None]"
8,5,cálculo,"[0, 1, 1, 2, 3, 5, None]"
9,4,reuso,"[0, 1, 1, 2, 3, 5, None]"


,Índice,Valor salvo
0,0,0
1,1,1
2,2,1
3,3,2
4,4,3
5,5,5
6,6,8


## Bottom-up com vetor

In [6]:
def fib_bottom_up(n):
    if n == 0:
        return 0, [0], 0

    F = [0] * (n + 1)
    F[0] = 0
    F[1] = 1
    iteracoes = 0

    for i in range(2, n + 1):
        iteracoes += 1
        F[i] = F[i - 1] + F[i - 2]

    return F[n], F, iteracoes


## Tabela bottom-up

In [7]:
linhas = []
for n in range(0, 11):
    valor, F, iteracoes = fib_bottom_up(n)
    linhas.append({'N': n, 'Fib(N)': valor, 'Iterações': iteracoes})

display(pd.DataFrame(linhas))


,N,Fib(N),Iterações
0,0,0,0
1,1,1,0
2,2,1,1
3,3,2,2
4,4,3,3
5,5,5,4
6,6,8,5
7,7,13,6
8,8,21,7
9,9,34,8


## Tabela de valores salvos no bottom-up

In [8]:
n = 10
valor, F, iteracoes = fib_bottom_up(n)

display(pd.DataFrame({'Índice': list(range(len(F))), 'Valor salvo': F}))


,Índice,Valor salvo
0,0,0
1,1,1
2,2,1
3,3,2
4,4,3
5,5,5
6,6,8
7,7,13
8,8,21
9,9,34


## Bottom-up otimizado

In [9]:
def fib_otimizado(n):
    if n <= 1:
        return n, 0

    anterior2 = 0
    anterior1 = 1
    iteracoes = 0

    for i in range(2, n + 1):
        iteracoes += 1
        atual = anterior1 + anterior2
        anterior2 = anterior1
        anterior1 = atual

    return anterior1, iteracoes


## Comparação geral

In [10]:
linhas = []
for n in range(0, 11):
    valor_rec, cont_rec = fib_recursivo(n)
    valor_memo, memo, cont_memo = fib_top_down(n)
    valor_bottom, F, iter_bottom = fib_bottom_up(n)
    valor_otim, iter_otim = fib_otimizado(n)

    linhas.append({
        'N': n,
        'Fib(N)': valor_rec,
        'Chamadas recursivo': cont_rec.chamadas,
        'Chamadas top-down': cont_memo.chamadas,
        'Iterações bottom-up': iter_bottom,
        'Iterações otimizado': iter_otim
    })

display(pd.DataFrame(linhas))


,N,Fib(N),Chamadas recursivo,Chamadas top-down,Iterações bottom-up,Iterações otimizado
0,0,0,1,1,0,0
1,1,1,1,1,0,0
2,2,1,3,3,1,1
3,3,2,5,5,2,2
4,4,3,9,7,3,3
5,5,5,15,9,4,4
6,6,8,25,11,5,5
7,7,13,41,13,6,6
8,8,21,67,15,7,7
9,9,34,109,17,8,8
